# HinEmo — Cleaning Pipeline
Reads the merged YouTube data (`data/interim/youtube_merged.csv`) and applies
Step 1.2 cleaning rules: dedup, min length, spam/URL removal. Code-mix ratio
filtering happens in a later notebook once language tags exist (IndicLID or
GPT-based) — this pass handles everything that doesn't need that yet.

In [1]:
import pandas as pd
import re
import os
import string
from collections import Counter

SOURCE_FILES = [
    "../data/raw/youtube_raw.csv",
    "../data/raw/youtube_generalized_raw.csv",
]

frames = []
for path in SOURCE_FILES:
    if os.path.exists(path):
        f = pd.read_csv(path)
        f["_source_file"] = path
        frames.append(f)
        print(f"Loaded {len(f)} rows from {path}")
    else:
        print(f"[!] {path} not found, skipping")

df = pd.concat(frames, ignore_index=True)
before = len(df)
df = df.drop_duplicates(subset=["source_id"], keep="first")
print(f"\nMerged: {before} -> {len(df)} unique rows ({before - len(df)} duplicates removed)")
df.head()

Loaded 52553 rows from ../data/raw/youtube_raw.csv
Loaded 50000 rows from ../data/raw/youtube_generalized_raw.csv

Merged: 102553 -> 102553 unique rows (0 duplicates removed)


,source_id,source,video_id,target_emotion,text,like_count,published_at,_source_file
0,UgzcT8JsA0dMHL-hl_94AaABAg,youtube_comment,dd8l2IZJaPU,anger,I've tried to explain MSP and economic viabili...,22817,2024-03-13T14:23:19Z,../data/raw/youtube_raw.csv
1,UgwHBaZa3bpEnAncy3h4AaABAg,youtube_comment,dd8l2IZJaPU,anger,इतिहास याद रखेगा की हजारों गिधड पत्रकारों के ब...,19126,2024-03-13T13:54:23Z,../data/raw/youtube_raw.csv
2,UgzHMMWzpDjhEZLpIYh4AaABAg,youtube_comment,dd8l2IZJaPU,anger,Full support for all farmers ❤,14929,2024-03-13T17:32:54Z,../data/raw/youtube_raw.csv
3,Ugxt2HKPwrkAmrqBFIZ4AaABAg,youtube_comment,dd8l2IZJaPU,anger,Tahalka machega phir se,44346,2024-03-13T14:32:21Z,../data/raw/youtube_raw.csv
4,Ugz9Ay6ZnY5edph2OCB4AaABAg,youtube_comment,dd8l2IZJaPU,anger,ਬਹੁਤ ਵਧੀਆ ਤਰੀਕੇ ਨਾਲ ਦਸਿਆ ਧਰੋ ਰਾਠੇ ਨੇ। ਇਸ ਨੂੰ ਬ...,75,2024-10-25T03:42:58Z,../data/raw/youtube_raw.csv


In [2]:
print("By target_emotion (blank = generalized/unlabeled):")
print(df["target_emotion"].fillna("(blank)").value_counts())

By target_emotion (blank = generalized/unlabeled):
target_emotion
(blank)      50000
happiness    10063
disgust       9152
sadness       9086
fear          8234
anger         8117
surprise      7901
Name: count, dtype: int64


In [3]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")

def basic_clean(text):
    text = URL_RE.sub("", str(text))
    text = MENTION_RE.sub("", text)
    return text.strip()

df["text_clean"] = df["text"].apply(basic_clean)
print(f"Created text_clean for {len(df)} rows")
df[["text", "text_clean"]].head()

Created text_clean for 102553 rows


,text,text_clean
0,I've tried to explain MSP and economic viabili...,I've tried to explain MSP and economic viabili...
1,इतिहास याद रखेगा की हजारों गिधड पत्रकारों के ब...,इतिहास याद रखेगा की हजारों गिधड पत्रकारों के ब...
2,Full support for all farmers ❤,Full support for all farmers ❤
3,Tahalka machega phir se,Tahalka machega phir se
4,ਬਹੁਤ ਵਧੀਆ ਤਰੀਕੇ ਨਾਲ ਦਸਿਆ ਧਰੋ ਰਾਠੇ ਨੇ। ਇਸ ਨੂੰ ਬ...,ਬਹੁਤ ਵਧੀਆ ਤਰੀਕੇ ਨਾਲ ਦਸਿਆ ਧਰੋ ਰਾਠੇ ਨੇ। ਇਸ ਨੂੰ ਬ...


In [4]:
def token_count(text):
    return len(text.split())

MIN_TOKENS = 5

before_length = len(df)
df = df[df["text_clean"].apply(token_count) >= MIN_TOKENS]
print(f"Before length filter: {before_length}")
print(f"After length filter (>= {MIN_TOKENS} tokens): {len(df)} (-{before_length - len(df)})")

Before length filter: 102553
After length filter (>= 5 tokens): 73749 (-28804)


In [5]:
df.sample(15)[["text_clean", "target_emotion"]]

,text_clean,target_emotion
95949,"किसी मज़दूर की हथेलियों का पसीना था,\nकिसी किस...",NaN
76585,Whoever design his home must have used their e...,NaN
23064,Bhai apna or pie ka dhyan rakhna ❤❤,sadness
76550,Brooooo this is surreal nd unbelievable soo be...,NaN
90022,Can you make a video on the rise of Sikh empir...,NaN
44462,then y parents teaching so much morel things a...,disgust
95081,Conclusion of this video\nMost powerful \nPais...,NaN
52869,If everyone becomes educated how will minister...,NaN
61086,4:10 ye sai tha Guru 😂😂,NaN
86148,Tabhi to inke marne pr janta ko koi dukh nhi h...,NaN


In [6]:
OTHER_SCRIPT_RANGES = {
    "gurmukhi_punjabi": (0x0A00, 0x0A7F),
    "bengali": (0x0980, 0x09FF),
    "tamil": (0x0B80, 0x0BFF),
    "telugu": (0x0C00, 0x0C7F),
    "kannada": (0x0C80, 0x0CFF),
    "malayalam": (0x0D00, 0x0D7F),
    "gujarati": (0x0A80, 0x0AFF),
    "odia": (0x0B00, 0x0B7F),
}

def detect_other_scripts(text):
    found = set()
    for char in str(text):
        cp = ord(char)
        for script, (start, end) in OTHER_SCRIPT_RANGES.items():
            if start <= cp <= end:
                found.add(script)
    return found

df["other_scripts"] = df["text_clean"].apply(detect_other_scripts)
df["has_other_script"] = df["other_scripts"].apply(lambda s: len(s) > 0)

print(f"Rows with non-Hindi Indic script content: {df['has_other_script'].sum()} "
      f"({df['has_other_script'].sum() / len(df):.1%})")

all_scripts = Counter()
for s in df["other_scripts"]:
    all_scripts.update(s)
print("\nBreakdown by script:")
for script, count in all_scripts.most_common():
    print(f"  {script}: {count}")

Rows with non-Hindi Indic script content: 451 (0.6%)

Breakdown by script:
  telugu: 172
  tamil: 117
  gurmukhi_punjabi: 61
  bengali: 61
  gujarati: 19
  odia: 12
  kannada: 7
  malayalam: 4


In [7]:
for script in all_scripts:
    subset = df[df["other_scripts"].apply(lambda s: script in s)]
    print(f"\n=== {script} sample ({len(subset)} total) ===")
    display(subset.sample(min(5, len(subset)))[["text_clean"]])


=== gurmukhi_punjabi sample (61 total) ===


,text_clean
40,ਧੰਨਵਾਦ ਕਿਸਾਨ ਦੀ ਗੱਲ ਵਧੀਆ ਢੰਗ ਨਾਲ ਰੱਖਣ ਲਈ। ❤❤🎉🎉🎉🎉
3006,Kisan majdoor ekta zindabaad \nਕਿਸਾਨ ਮਜ਼ਦੂਰ ਏਤ...
86995,ਭੂਤ ਵੀ ਹੁੰਦੇ ਨੇ ਐਂਨੇ ਸੋਹਣੇ ਸੋਹਣੇ \nਜੱਟ ਪਟੇਲ ਮਾ...
90995,ਚਾਈਨਾ ਦਾ ਇਹ ਸਮਝੋਤਾ ਪੰਜਾਬ ਦੇ ਸਿੱਖਾਂ ਨਾਲ ਹੋਈ ਸੀ ...
5080,"#ਲੇਫ਼੍ਟ ਪ੍ਰਤੀਏਸ , ਖਾਲਿਸਤਾਨੀ , ਸ਼ੀਂ ਬਾਗ਼ , ਕਾਂਗਰਸ ..."



=== bengali sample (61 total) ===


,text_clean
94749,একদিন ভারতের যে সুনাম ছিলো এখন সেটার জারিজুরি ...
38068,আমিও একদিন এইভাবে বাবা মা কে জড়িয়ে ধরে আনন্দ...
21051,খুব পিইন ফুল একটা কথা
1094,আরে মোদী ভাই তুমি বোলনা হে ঠা মুছকে বোলিয়ে বা...
89371,নেহেরু সব বুঝেও চুপ থেকেছে কারণ কংগ্রেস দুর্বল...



=== malayalam sample (4 total) ===


...

,text_clean
43254,తప్పుగా ఉంది మెసేజ్... పోలీస్ శాఖ 100% అవినీతి...
40871,ఎంత మంది ఎక్కినరు లెక్క పత్రం లేదా
41193,అబ్బ అబ్బ కళ్ళముందు హృదయ విధారకం వాళ్లందరిఖి స...
63568,जे बात चौकीदार ही चोर है\nఅసలు విషయం అదే—కాపలా...
41175,బోటులో ప్రయాణించే వారు తప్పకుండా లైఫ్ సేవర్ జా...



=== kannada sample (7 total) ===


,text_clean
49672,"There is a saying in kannada, ಕುಂಬಳಕಾಯಿ ಕಳ್ಳ ಅ..."
55519,"3:00 ಒಮ್ಮೆ ಗೆದ್ದು ಅಧಿಕಾರ ಸೂತ್ರ ಹಿಡಿದರಾಯ್ತು, ನಾ..."
10259,ರಾಬರ್ಟ್ ರಿವೀವ್ ಮಾಡಲೇ ಯಪ್ಪಾ.... ಕೋಳಿ ತಿಪ್ಪ .......
28269,", ಎಂತ ಮಾರ್ಗದ ಬಗ್ಗೆ ಎಲ್ಲ ಹೆದರ್ತಾರೆ😂"
61591,8:16 ಯಾಕೋ ಕನ್ನಡ ಮತ್ ಹಾಡೋ 😡😡😡



=== odia sample (12 total) ===


,text_clean
16402,"In Bengal, Bengalis welcome ancestors by keepi..."
16501,I love ଓଡିଶା ❤❤❤❤❤❤❤❤❤❤❤❤❤ Happy diwali
71542,ରଜା ଡୋଲି .... ମୋହନ ମା ji ୀ ବଙ୍ଗାଳୀ |
16368,ଓଡ଼ିଶାରେ ଦିପାବଳୀ ଏକ ବଡ଼ ପୂଜା 🙏
16163,ଓଡ଼ିଶାରେ ଦିପାବଳୀ ରେ ବଡ଼ବଡ଼ିଆ ଡାକ👏✨


In [8]:
before_script_filter = len(df)
df = df[~df["has_other_script"]].copy()
print(f"Before: {before_script_filter}")
print(f"After dropping other-script rows: {len(df)} (-{before_script_filter - len(df)})")

df = df.drop(columns=["other_scripts", "has_other_script"])

df.to_csv("../data/interim/youtube_cleaned_stage1.csv", index=False)
print(f"\nSaved checkpoint: {len(df)} rows -> data/interim/youtube_cleaned_stage1.csv")

Before: 73749
After dropping other-script rows: 73298 (-451)

Saved checkpoint: 73298 rows -> data/interim/youtube_cleaned_stage1.csv


In [12]:
import pandas as pd

df_stage1 = pd.read_csv("../data/interim/youtube_cleaned_stage1.csv")
print(f"Loaded {len(df_stage1)} rows from checkpoint")

Loaded 73298 rows from checkpoint


/var/folders/m8/j965yrg954b9_zblcxyfkbhm0000gn/T/ipykernel_91818/4074694878.py:3: DtypeWarning: Columns (0: target_emotion) have mixed types. Specify dtype option on import or set low_memory=False.
  df_stage1 = pd.read_csv("../data/interim/youtube_cleaned_stage1.csv")


In [14]:
import string

def has_latin_letters(text):
    return any(c in string.ascii_letters for c in str(text))

df_stage1["has_latin"] = df_stage1["text_clean"].apply(has_latin_letters)
pure_devanagari = df_stage1[~df_stage1["has_latin"]]

print(f"Pure-Devanagari (no Latin characters at all): {len(pure_devanagari)} "
      f"({len(pure_devanagari) / len(df_stage1):.1%})")

Pure-Devanagari (no Latin characters at all): 3348 (4.6%)


In [15]:
pure_devanagari.sample(min(15, len(pure_devanagari)))[["text_clean"]]

,text_clean
2657,भाई तू ही था जो सूर्य कुमार यादव को निकालने का...
36489,गुजरात अहमदाबाद में फंपनीयो फेकटरीया के केमिकल...
37559,अपनी डूबती साख बचाने के लिए फैजल खान आजकल पटना...
39333,झारखंड में संतालों के खिलाफ जारी भूमि विस्थापन...
45178,अंधभक्तों अपने जीजा की बात को तो मान लो।\nया इ...
23287,अरे बहुत छोड़ो किसी का नाम आया है या नहीं आया ...
48110,लेकिन भगवान ने कब कहा उसको पैसे चाहिए 😂
14572,दोनों ही नहीं हैं 👌👌👍
68921,नीतीश जी आपको फिर से नया वीडियो बनाना पड़ेगा इ...
39252,मै बिना वीडियो देखे कह सकता हु की हो न हो इसमे...


In [16]:
df_stage2 = df_stage1[df_stage1["has_latin"]].drop(columns=["has_latin"]).copy()

print(f"Before: {len(df_stage1)}")
print(f"After dropping pure-Devanagari: {len(df_stage2)} (-{len(df_stage1) - len(df_stage2)})")

df_stage2.to_csv("../data/interim/youtube_cleaned_stage2.csv", index=False)
print(f"\nSaved checkpoint: {len(df_stage2)} rows -> data/interim/youtube_cleaned_stage2.csv")

Before: 73298
After dropping pure-Devanagari: 69950 (-3348)

Saved checkpoint: 69950 rows -> data/interim/youtube_cleaned_stage2.csv


In [17]:
print(df["target_emotion"].fillna("(blank)").value_counts())

target_emotion
(blank)      36755
disgust       7820
happiness     6892
anger         6281
surprise      5399
sadness       5251
fear          4900
Name: count, dtype: int64


In [ ]:
git st